# Utils function used sometimes on the fly to tests stuff.

In [ ]:
import torchaudio
import torch
from pathlib import Path
from datasets import concatenate_datasets

In [ ]:

def split_wav(file_path, out_dir, chunk_duration=0.5, target_sr=16000):
    """
    Split a WAV file into fixed-length chunks and save them.

    Parameters:
        file_path (str or Path): Path to input WAV file.
        out_dir (str or Path): Directory to save chunks.
        chunk_duration (float): Duration of each chunk in seconds (default 0.5s).
        target_sr (int): Target sampling rate (default 16000 Hz).
    """
    file_path = Path(file_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Load audio
    waveform, sr = torchaudio.load(file_path)

    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # Resample if needed
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(sr, target_sr)
        waveform = resampler(waveform)
        sr = target_sr

    num_samples_per_chunk = int(chunk_duration * sr)
    total_samples = waveform.shape[1]
    num_chunks = (total_samples + num_samples_per_chunk - 1) // num_samples_per_chunk

    # Split and save
    for i in range(num_chunks):
        start = i * num_samples_per_chunk
        end = min(start + num_samples_per_chunk, total_samples)
        chunk = waveform[:, start:end]

        # Pad last chunk if too short
        if chunk.shape[1] < num_samples_per_chunk:
            padding = num_samples_per_chunk - chunk.shape[1]
            chunk = torch.nn.functional.pad(chunk, (0, padding))

        out_file = out_dir / f"{file_path.stem}_chunk{i+1}.wav"
        torchaudio.save(str(out_file), chunk, sr)

    print(f"✅ Saved {num_chunks} chunks to {out_dir}")


In [ ]:
# split_wav("/home/pierre/Downloads/drone-sound-128-ytshorts.savetube.me.wav", "/home/pierre/Documents/Projects/PST4/AI/data/raw/test/bb")
# split_wav("/home/pierre/Downloads/this-is-what-a-drone-sounds-like-128-ytshorts.savetube.me.wav", "/home/pierre/Documents/Projects/PST4/AI/data/raw/test/aa")
# split_wav("/home/pierre/Downloads/this-drone-flight-is-unreal-128-ytshorts.savetube.me.wav", "/home/pierre/Documents/Projects/PST4/AI/data/raw/test/aa")
# split_wav("/home/pierre/Downloads/drone-sound-effects-128-ytshorts.savetube.me.mp3", "/home/pierre/Documents/Projects/PST4/AI/data/raw/test/aa")
# split_wav("/home/pierre/Downloads/chilling-sound-of-a-uaf-drone-hunting-for-a-cowering-russian-soldier-128-ytshorts.savetube.me.mp3", "/home/pierre/Documents/Projects/PST4/AI/data/raw/test/aa")
# split_wav("/home/pierre/Documents/Projects/PST4/AI/data/raw/rec.wav", "/home/pierre/Documents/Projects/PST4/AI/data/raw/test/cc")
# split_wav("/home/pierre/Downloads/Download 2026-01-28T20_18_18/dronee.wav", "/home/pierre/Documents/Projects/PST4/AI/data/raw/test/28-01-2026/drone1")
# split_wav("/home/pierre/Downloads/Download 2026-01-28T20_18_18/test.wav", "/home/pierre/Documents/Projects/PST4/AI/data/raw/test/28-01-2026/other1/other")
# split_wav("/home/pierre/Documents/Projects/PST4/AI/data/raw/test/30-01-2026/drone0.wav", "/home/pierre/Documents/Projects/PST4/AI/data/raw/test/30-01-2026/temp")
# split_wav("/home/pierre/Downloads/Download 2026-02-02T19_14_29/longrec12.wav", "/home/pierre/Documents/Projects/PST4/AI/data/raw/test/02-02-2026/other")
split_wav("/home/pierre/Documents/Projects/PST4/AI/data/raw/record/other/rec.wav", "/home/pierre/Documents/Projects/PST4/AI/data/raw/test/07-02-2026/other")

In [ ]:
import os

def process_wav_folder(input_dir, output_dir):
    """
    Processes all .wav files in a given folder using split_wav(file_path, out_dir).

    Args:
        input_dir (str): Path to the folder containing .wav files.
        output_dir (str): Path to the folder where processed files will be saved.
    """

    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    for filename in os.listdir(input_dir):
        if filename.lower().endswith(".wav"):
            file_path = os.path.join(input_dir, filename)
            print(f"Processing: {file_path}")

            try:
                split_wav(file_path, output_dir)
            except Exception as e:
                print(f"❌ Error processing {filename}: {e}")

    print("✅ All .wav files processed successfully!")



In [ ]:
# process_wav_folder("/home/pierre/Documents/Projects/PST4/AI/data/raw/new/drone_test", "/home/pierre/Documents/Projects/PST4/AI/data/raw/new/drone_test_output")

In [ ]:
from datasets import load_dataset, ClassLabel, DatasetDict, concatenate_datasets

ds = load_dataset("audiofolder", data_dir="../../data/raw/new/drone_output")

In [ ]:
labels = ds["train"].features["label"]

In [ ]:
file_names = [p.split("/")[-1] for p in ds["train"].__dict__["_info"].download_checksums.keys()]
ds["train"] = ds["train"].add_column("filename", file_names)

In [ ]:
ds

In [ ]:
from IPython.display import display, Audio
ad1 = ds["train"][1]
print(labels.int2str(ad1["label"]), ad1["filename"])
display(Audio(ad1["audio"]["array"], rate=ad1["audio"]["sampling_rate"]))

In [ ]:
old_dataset = load_dataset("Usernameeeeee/df_462700_2")

In [ ]:
print(ds)
print(old_dataset)

In [ ]:
new_dataset = concatenate_datasets([ds["train"], old_dataset["train"]])

In [ ]:
ds.push_to_hub("Usernameeeeee/drone_test_2")

In [ ]:
from datasets import load_dataset
from collections import Counter
dataset = load_dataset("Usernameeeeee/drone_test_2")

In [ ]:
label_counter = Counter()
print(dataset["test"].features["label"])
Counter(dataset["test"]["label"])
